In [1]:
import pandas as pd
import sqlalchemy as sa
import urllib
from sqlalchemy import create_engine
from sqlalchemy import text
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv('C:/Users/bryan/OneDrive/Desktop/Power_BI_Dashboards/Chocolate Sales Dashboard/Raw_data/Chocolate_Sales.csv')

In [2]:
df.insert(0, 'order_id', range(1, 1 + len(df))) ##insert index primary key
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True) ## Convert to datetime indicating format
df['sales_month'] = df['Date'].dt.month # extract month from datetime
df['sales_year'] = df['Date'].dt.year # extract year from datetime
df['Amount'] = df['Amount'].str.replace(r'[$,]', '', regex=True) # takeout '$' in string to be able to convert datatype
df['Amount'] = df['Amount'].astype(float) 
df['Amount'] = df['Amount'].astype(int)# data conversion to int
df['price_per_box'] = df['Amount']/df['Boxes Shipped'].astype(int) # calculate price per box



In [3]:
# 1. Setup SQL connection
server = r'Chris\TEST' 
database = 'Test'  # Database

# 2. Build the connection parameters for Windows Auth
params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)

# 1. Create the engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 2. Define the view name for consistency
view_name = "v_ChocolateSalesAnalysis"

# 3. Test connection, Push Data, and Refresh View
try:
    # Push the cleaned DataFrame to the SQL table
    df.to_sql('chocolate_sales', con=engine, if_exists='replace', index=False)
    print("Success! Table 'chocolate_sales' updated.")

    # Open a connection to handle the View refresh
    with engine.connect() as conn:
        # Drop the existing view to clear old metadata
        conn.execute(text(f"IF OBJECT_ID('{view_name}', 'V') IS NOT NULL DROP VIEW {view_name}"))
        
        # Recreate the view to capture all current columns
        conn.execute(text(f"CREATE VIEW {view_name} AS SELECT * FROM dbo.chocolate_sales"))
        
        # Commit the transaction (required in some SQLAlchemy versions)
        conn.commit() 
        
    print(f"Success! View '{view_name}' is now in sync with your latest columns.")

except Exception as e:
    print(f"Error during ETL process: {e}")

c:\Users\bryan\.conda\envs\project\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Success! Table 'chocolate_sales' updated.
Success! View 'v_ChocolateSalesAnalysis' is now in sync with your latest columns.
